In [1]:
import sys
import os
from math import log
import numpy as np
import seaborn as sns
import pandas as pd
import scipy as sp
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow import keras
from tensorflow.keras import layers
import cv2 as cv
import pickle
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import compute_class_weight
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_curve, roc_auc_score, auc

np.random.seed(72)
tf.random.set_seed(72)
sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(log_device_placement=True))

D:\anaconda\envs\tf_gpu\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.3.0 and strictly below 2.6.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/addons
  warnings.warn(


Device mapping:
/job:localhost/replica:0/task:0/device:GPU:0 -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9



In [2]:
def get_train_test(train_dir, test_dir):
    datagen_train = ImageDataGenerator(rescale=1./255, width_shift_range=0.1, height_shift_range=0.1,
                                      horizontal_flip=True,  vertical_flip=False)
    datagen_test = ImageDataGenerator(rescale=1./255)

    generator_train = datagen_train.flow_from_directory(directory=train_dir, target_size=(image_size, image_size),
                                                        batch_size=batch_size, shuffle=True)
    generator_test = datagen_test.flow_from_directory(directory=test_dir, target_size=(image_size, image_size),
                                                      batch_size=batch_size, shuffle=False)

    num_classes = generator_train.num_classes
    labels = generator_train.classes
    class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
    class_weights = dict(zip(np.unique(labels), class_weights))
    print(class_weights)
    
    steps_per_epoch = generator_train.n / batch_size
    steps_test = generator_test.n / batch_size
    return generator_train, generator_test, labels, class_weights, num_classes, steps_per_epoch, steps_test

def compile_model(model, weight):
    optimizer = Adam(lr=0.0000026) 
    loss = 'categorical_crossentropy'

    metrics = ['accuracy', 'categorical_accuracy', tf.keras.metrics.AUC(), tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), 
               tf.keras.metrics.TruePositives(), tf.keras.metrics.TrueNegatives(), tf.keras.metrics.FalsePositives(), 
               tf.keras.metrics.FalseNegatives(), tfa.metrics.CohenKappa(num_classes = num_classes), 
               tfa.metrics.F1Score(num_classes = num_classes)]

    model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

    lr = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.1,
        patience=9, mode="max", min_delta=0.0001, min_lr=0.00001, verbose=1)
    checkpoint = ModelCheckpoint(filepath=weight, save_best_only=True, monitor = 'val_accuracy', verbose=1)
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=0, patience=10, verbose=1, restore_best_weights=True)

    callbacks = [lr, checkpoint, early_stopping]
    return model, callbacks

def evaluate_(model, generator_test):
    model.evaluate(generator_test)
    
    y_pred = model.predict(generator_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = generator_test.classes
    class_labels = list(generator_test.class_indices.keys())

    print(classification_report(y_true, y_pred_classes))
    cm = confusion_matrix(y_true, y_pred_classes)
    
    # Plotting the confusion matrix
    plt.figure(figsize=(8, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
    plt.show()
    
def plot_graphs(history, generator_test):
    plt.figure(figsize=(15,7))
    plt.plot(history.history['accuracy'], 'r', linewidth=2.5)
    plt.plot(history.history['val_accuracy'], linewidth=2.5)
    plt.title('Model Accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['Training Accuracy', 'Testing Accuracy'], loc='upper left')
    plt.show()

    plt.figure(figsize=(15,7))
    plt.plot(history.history['loss'], 'r', linewidth=2.5)
    plt.plot(history.history['val_loss'], linewidth=2.5)
    plt.title('Model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['Train Loss', 'Test Loss'], loc='upper left')
    plt.show()
    
    # Generate predictions
    predictions = model.predict_generator(generator_test)
    # Get true labels
    true_labels = generator_test.classes
    # Binarize the true labels
    encoder = OneHotEncoder(sparse=False)
    true_labels = encoder.fit_transform(true_labels.reshape(-1, 1))
    # Calculate ROC curve and ROC AUC for each class
    n_classes = true_labels.shape[1]
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for class_idx in range(n_classes):
        fpr[class_idx], tpr[class_idx], _ = roc_curve(true_labels[:, class_idx], predictions[:, class_idx])
        roc_auc[class_idx] = auc(fpr[class_idx], tpr[class_idx])

    # Plot ROC curves for each class
    plt.figure(figsize=(12, 12))
    for class_idx in range(n_classes):

        plt.plot(fpr[class_idx], tpr[class_idx], label='ROC curve (area = %0.2f) for class %d' % (roc_auc[class_idx], class_idx),
                linewidth=2.5)

    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic per Class')
    plt.legend(loc='lower right')
    plt.show()

In [3]:
# Params
num_classes = 2
batch_size = 8
image_size = 112
weight = 'NFNetF2.h5'

In [4]:
from keras_cv_attention_models import nfnets
model = nfnets.NFNetF2(input_shape=(image_size, image_size, 3) , num_classes=num_classes, 
                         drop_connect_rate=0.2, classifier_activation="softmax")

>>>> Load pretrained from: C:\Users\sakib\.keras\models\nfnetf2_imagenet.h5


In [5]:
train_dir = r"F:\Projects\Personal\Ocular Toxoplasmosis\data\BinaryClassification\train"
test_dir = r"F:\Projects\Personal\Ocular Toxoplasmosis\data\BinaryClassification\val"
generator_train, generator_test, labels, class_weights, num_classes, steps_per_epoch, steps_test = get_train_test(train_dir, test_dir)
model, callbacks = compile_model(model, weight)

Found 366 images belonging to 2 classes.
Found 83 images belonging to 2 classes.
{0: 1.3863636363636365, 1: 0.782051282051282}


D:\anaconda\envs\tf_gpu\lib\site-packages\keras\optimizer_v2\adam.py:105: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


In [6]:
model.summary()

Model: "nfnetf2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 112, 112, 3  0           []                               
                                )]                                                                
                                                                                                  
 stem_1_conv (ScaledStandardize  (None, 56, 56, 16)  464         ['input_1[0][0]']                
 dConv2D)                                                                                         
                                                                                                  
 stem_1_gelu (Activation)       (None, 56, 56, 16)   0           ['stem_1_conv[0][0]']            
                                                                                            

 stack1_block1_se_sigmoid (Acti  (None, 1, 1, 256)   0           ['stack1_block1_se_2_conv[0][0]']
 vation)                                                                                          
                                                                                                  
 stack1_block1_se_out (Multiply  (None, 28, 28, 256)  0          ['stack1_block1_deep_4_conv[0][0]
 )                                                               ',                               
                                                                  'stack1_block1_se_sigmoid[0][0]'
                                                                 ]                                
                                                                                                  
 tf.math.multiply_8 (TFOpLambda  (None, 28, 28, 256)  0          ['stack1_block1_se_out[0][0]']   
 )                                                                                                
          

 tf.math.multiply_15 (TFOpLambd  (None, 28, 28, 256)  0          ['stack1_block2_se_out[0][0]']   
 a)                                                                                               
                                                                                                  
 dropout (Dropout)              (None, 28, 28, 256)  0           ['tf.math.multiply_15[0][0]']    
                                                                                                  
 stack1_block2_deep_gain (ZeroI  (None, 28, 28, 256)  1          ['dropout[0][0]']                
 nitGain)                                                                                         
                                                                                                  
 tf.math.multiply_16 (TFOpLambd  (None, 28, 28, 256)  0          ['stack1_block2_deep_gain[0][0]']
 a)                                                                                               
          

                                                                                                  
 stack1_block3_output (Add)     (None, 28, 28, 256)  0           ['stack1_block2_output[0][0]',   
                                                                  'tf.math.multiply_23[0][0]']    
                                                                                                  
 stack2_block1_preact_gelu (Act  (None, 28, 28, 256)  0          ['stack1_block3_output[0][0]']   
 ivation)                                                                                         
                                                                                                  
 tf.math.multiply_24 (TFOpLambd  (None, 28, 28, 256)  0          ['stack2_block1_preact_gelu[0][0]
 a)                                                              ']                               
                                                                                                  
 tf.math.m

                                                                                                  
 stack2_block2_preact_gelu (Act  (None, 14, 14, 512)  0          ['stack2_block1_output[0][0]']   
 ivation)                                                                                         
                                                                                                  
 tf.math.multiply_31 (TFOpLambd  (None, 14, 14, 512)  0          ['stack2_block2_preact_gelu[0][0]
 a)                                                              ']                               
                                                                                                  
 tf.math.multiply_32 (TFOpLambd  (None, 14, 14, 512)  0          ['tf.math.multiply_31[0][0]']    
 a)                                                                                               
                                                                                                  
 stack2_bl

...

 tf.math.multiply_221 (TFOpLamb  (None, 4, 4, 1536)  0           ['tf.math.multiply_220[0][0]']   
 da)                                                                                              
                                                                                                  
 stack4_block5_deep_1_conv (Sca  (None, 4, 4, 768)   1181184     ['tf.math.multiply_221[0][0]']   
 ledStandardizedConv2D)                                                                           
                                                                                                  
 stack4_block5_deep_1_gelu (Act  (None, 4, 4, 768)   0           ['stack4_block5_deep_1_conv[0][0]
 ivation)                                                        ']                               
                                                                                                  
 tf.math.multiply_222 (TFOpLamb  (None, 4, 4, 768)   0           ['stack4_block5_deep_1_gelu[0][0]
 da)      

 da)                                                             ']                               
                                                                                                  
 stack4_block6_deep_2_conv (Sca  (None, 4, 4, 768)   886272      ['tf.math.multiply_229[0][0]']   
 ledStandardizedConv2D)                                                                           
                                                                                                  
 stack4_block6_deep_2_gelu (Act  (None, 4, 4, 768)   0           ['stack4_block6_deep_2_conv[0][0]
 ivation)                                                        ']                               
                                                                                                  
 tf.math.multiply_230 (TFOpLamb  (None, 4, 4, 768)   0           ['stack4_block6_deep_2_gelu[0][0]
 da)                                                             ']                               
          

                                                                                                  
 stack4_block7_deep_3_conv (Sca  (None, 4, 4, 768)   886272      ['tf.math.multiply_237[0][0]']   
 ledStandardizedConv2D)                                                                           
                                                                                                  
 stack4_block7_deep_3_gelu (Act  (None, 4, 4, 768)   0           ['stack4_block7_deep_3_conv[0][0]
 ivation)                                                        ']                               
                                                                                                  
 tf.math.multiply_238 (TFOpLamb  (None, 4, 4, 768)   0           ['stack4_block7_deep_3_gelu[0][0]
 da)                                                             ']                               
                                                                                                  
 stack4_bl

 stack4_block8_deep_4_conv (Sca  (None, 4, 4, 1536)  1182720     ['tf.math.multiply_245[0][0]']   
 ledStandardizedConv2D)                                                                           
                                                                                                  
 tf.math.reduce_mean_34 (TFOpLa  (None, 1, 1, 1536)  0           ['stack4_block8_deep_4_conv[0][0]
 mbda)                                                           ']                               
                                                                                                  
 stack4_block8_se_1_conv (Conv2  (None, 1, 1, 768)   1180416     ['tf.math.reduce_mean_34[0][0]'] 
 D)                                                                                               
                                                                                                  
 stack4_block8_se_relu (Activat  (None, 1, 1, 768)   0           ['stack4_block8_se_1_conv[0][0]']
 ion)     

 ion)                                                                                             
                                                                                                  
 stack4_block9_se_2_conv (Conv2  (None, 1, 1, 1536)  1181184     ['stack4_block9_se_relu[0][0]']  
 D)                                                                                               
                                                                                                  
 stack4_block9_se_sigmoid (Acti  (None, 1, 1, 1536)  0           ['stack4_block9_se_2_conv[0][0]']
 vation)                                                                                          
                                                                                                  
 stack4_block9_se_out (Multiply  (None, 4, 4, 1536)  0           ['stack4_block9_deep_4_conv[0][0]
 )                                                               ',                               
          

In [ ]:
epochs = 50
history = model.fit_generator(generator=generator_train, epochs=epochs,steps_per_epoch=steps_per_epoch,
                                  validation_data=generator_test, validation_steps=steps_test,
                                 callbacks= callbacks, class_weight =class_weights)

C:\Users\sakib\AppData\Local\Temp\ipykernel_22168\3578285518.py:2: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(generator=generator_train, epochs=epochs,steps_per_epoch=steps_per_epoch,


Epoch 1/50
46/45 [==============================] - ETA: 0s - loss: 0.6968 - accuracy: 0.5628 - categorical_accuracy: 0.5628 - auc: 0.6009 - precision: 0.5628 - recall: 0.5628 - true_positives: 206.0000 - true_negatives: 206.0000 - false_positives: 160.0000 - false_negatives: 160.0000 - cohen_kappa: -0.9215 - f1_score: 0.5442

D:\anaconda\envs\tf_gpu\lib\site-packages\keras\engine\training.py:2034: UserWarning: Metric CohenKappa implements a `reset_states()` method; rename it to `reset_state()` (without the final "s"). The name `reset_states()` has been deprecated to improve API consistency.
  m.reset_state()
D:\anaconda\envs\tf_gpu\lib\site-packages\keras\engine\training.py:2034: UserWarning: Metric F1Score implements a `reset_states()` method; rename it to `reset_state()` (without the final "s"). The name `reset_states()` has been deprecated to improve API consistency.
  m.reset_state()



Epoch 1: val_accuracy improved from -inf to 0.57831, saving model to NFNetF2.h5
45/45 [==============================] - 52s 484ms/step - loss: 0.6968 - accuracy: 0.5628 - categorical_accuracy: 0.5628 - auc: 0.6009 - precision: 0.5628 - recall: 0.5628 - true_positives: 206.0000 - true_negatives: 206.0000 - false_positives: 160.0000 - false_negatives: 160.0000 - cohen_kappa: -0.9215 - f1_score: 0.5442 - val_loss: 0.6882 - val_accuracy: 0.5783 - val_categorical_accuracy: 0.5783 - val_auc: 0.6011 - val_precision: 0.5783 - val_recall: 0.5783 - val_true_positives: 48.0000 - val_true_negatives: 48.0000 - val_false_positives: 35.0000 - val_false_negatives: 35.0000 - val_cohen_kappa: -0.8902 - val_f1_score: 0.5523 - lr: 2.6000e-06
Epoch 2/50
46/45 [==============================] - ETA: 0s - loss: 0.5991 - accuracy: 0.6667 - categorical_accuracy: 0.6667 - auc: 0.7168 - precision: 0.6667 - recall: 0.6667 - true_positives: 244.0000 - true_negatives: 244.0000 - false_positives: 122.0000 - false_

46/45 [==============================] - ETA: 0s - loss: 0.1165 - accuracy: 0.9590 - categorical_accuracy: 0.9590 - auc: 0.9909 - precision: 0.9590 - recall: 0.9590 - true_positives: 351.0000 - true_negatives: 351.0000 - false_positives: 15.0000 - false_negatives: 15.0000 - cohen_kappa: -0.8738 - f1_score: 0.9561
Epoch 9: val_accuracy did not improve from 0.91566
45/45 [==============================] - 15s 321ms/step - loss: 0.1165 - accuracy: 0.9590 - categorical_accuracy: 0.9590 - auc: 0.9909 - precision: 0.9590 - recall: 0.9590 - true_positives: 351.0000 - true_negatives: 351.0000 - false_positives: 15.0000 - false_negatives: 15.0000 - cohen_kappa: -0.8738 - f1_score: 0.9561 - val_loss: 0.3146 - val_accuracy: 0.8795 - val_categorical_accuracy: 0.8795 - val_auc: 0.9427 - val_precision: 0.8795 - val_recall: 0.8795 - val_true_positives: 73.0000 - val_true_negatives: 73.0000 - val_false_positives: 10.0000 - val_false_negatives: 10.0000 - val_cohen_kappa: -0.9004 - val_f1_score: 0.8729 

Epoch 17/50
46/45 [==============================] - ETA: 0s - loss: 0.0469 - accuracy: 0.9836 - categorical_accuracy: 0.9836 - auc: 0.9981 - precision: 0.9836 - recall: 0.9836 - true_positives: 360.0000 - true_negatives: 360.0000 - false_positives: 6.0000 - false_negatives: 6.0000 - cohen_kappa: -0.8662 - f1_score: 0.9823
Epoch 17: val_accuracy did not improve from 0.95181
45/45 [==============================] - 15s 326ms/step - loss: 0.0469 - accuracy: 0.9836 - categorical_accuracy: 0.9836 - auc: 0.9981 - precision: 0.9836 - recall: 0.9836 - true_positives: 360.0000 - true_negatives: 360.0000 - false_positives: 6.0000 - false_negatives: 6.0000 - cohen_kappa: -0.8662 - f1_score: 0.9823 - val_loss: 0.3362 - val_accuracy: 0.9036 - val_categorical_accuracy: 0.9036 - val_auc: 0.9550 - val_precision: 0.9036 - val_recall: 0.9036 - val_true_positives: 75.0000 - val_true_negatives: 75.0000 - val_false_positives: 8.0000 - val_false_negatives: 8.0000 - val_cohen_kappa: -0.8797 - val_f1_score: 

In [ ]:
# model = keras.models.load_model(weight)
evaluate_(model, generator_test)
plot_graphs(history, generator_test)